In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ====================================================
# MODEL 1: THE DEEP-MEL ENCODER (CNN FROM SCRATCH)
# ====================================================

import os, random, librosa, torch, wandb
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm

# ======================
# W&B OFFLINE SETUP
# ======================
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_MODE"] = "offline"

wandb.init(
    project="DL-GenAi-t1-2026",
    name="Model1_DeepMel_Final",
    mode="offline"
)
print("✅ W&B initialized in OFFLINE mode")

# ======================
# PATHS
# ======================
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_PATH = os.path.join(BASE_PATH, "genres_stems")
ESC50_DIR = os.path.join(BASE_PATH, "ESC-50-master/audio")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# DATA EXPLORATION (YOUR CODE)
# ======================
durations, sample_rates, classes = [], [], []

for genre in os.listdir(TRAIN_PATH):
    genre_path = os.path.join(TRAIN_PATH, genre)
    classes.append(genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        for stem in ["drums.wav","bass.wav","vocals.wav","other.wav"]:
            file_path = os.path.join(song_path, stem)
            y, sr = librosa.load(file_path, sr=None)
            durations.append(len(y)/sr)
            sample_rates.append(sr)

print("Total files:", len(durations))
print("Unique sample rates:", set(sample_rates))
print("Average duration:", np.mean(durations))
print("Class distribution:", {g:len(os.listdir(os.path.join(TRAIN_PATH,g))) for g in classes})

# ======================
# DATA PREPARATION
# ======================
genres = sorted(os.listdir(TRAIN_PATH))
label_map = {g: i for i, g in enumerate(genres)}
rev_label_map = {i: g for g, i in label_map.items()}

all_data = []
for g in genres:
    g_path = os.path.join(TRAIN_PATH, g)
    for song in os.listdir(g_path):
        s_path = os.path.join(g_path, song)
        stems = {
            "drums": os.path.join(s_path, "drums.wav"),
            "bass": os.path.join(s_path, "bass.wav"),
            "vocals": os.path.join(s_path, "vocals.wav"),
            "other": os.path.join(s_path, "other.wav")
        }
        all_data.append((stems, g))

print("Total songs:", len(all_data))

train_list, val_list = train_test_split(
    all_data,
    test_size=0.1,
    stratify=[x[1] for x in all_data],
    random_state=42
)

# ======================
# DATASET
# ======================
class DeepMashupDataset(Dataset):
    def __init__(self, data_list, esc_path, is_train=True):
        self.data = data_list
        self.is_train = is_train
        self.noise_files = [os.path.join(esc_path, f) for f in os.listdir(esc_path)]
        self.pool = {g: [x[0] for x in data_list if x[1] == g] for g in genres}

    def __len__(self):
        return len(self.data)

    def _load(self, path):
        y, _ = librosa.load(path, sr=22050)
        target = 22050 * 30
        return np.pad(y, (0, max(0, target - len(y))))[:target]

    def __getitem__(self, idx):
        stems, g_name = self.data[idx]

        # Stem Mixing
        if self.is_train and random.random() < 0.7:
            source = random.choice(self.pool[g_name])
            mix = sum([self._load(source[k]) * random.uniform(0.8, 1.2)
                       for k in ["drums", "bass", "vocals", "other"]])
        else:
            mix = sum([self._load(v) for v in stems.values()])

        # Noise Injection
        if self.is_train:
            for _ in range(random.randint(1, 2)):
                nz = self._load(random.choice(self.noise_files))
                mix += nz * random.uniform(0.1, 0.3)

        # Mel Spectrogram
        mel = librosa.feature.melspectrogram(
            y=mix, sr=22050, n_mels=128, n_fft=2048, hop_length=512
        )
        mel = librosa.power_to_db(mel, ref=np.max)

        # SpecAugment
        if self.is_train:
            for _ in range(2):
                mel[random.randint(0, 118):, :] = mel.mean()
                mel[:, random.randint(0, mel.shape[1]-20):] = mel.mean()

        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        return torch.tensor(mel).unsqueeze(0).float(), label_map[g_name]

# ======================
# MODEL
# ======================
class DeepMelEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, 3, padding=1),
                nn.BatchNorm2d(out_f), nn.ReLU(),
                nn.Conv2d(out_f, out_f, 3, padding=1),
                nn.BatchNorm2d(out_f), nn.ReLU(),
                nn.MaxPool2d(2)
            )

        self.features = nn.Sequential(
            block(1, 64),
            block(64, 128),
            block(128, 256),
            block(256, 512),
            block(512, 512)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# ======================
# TRAINING
# ======================
model = DeepMelEncoder().to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

train_loader = DataLoader(DeepMashupDataset(train_list, ESC50_DIR, True),
                          batch_size=16, shuffle=True, num_workers=2)

val_loader = DataLoader(DeepMashupDataset(val_list, ESC50_DIR, False),
                        batch_size=16)

best_f1 = 0

for epoch in range(15):

    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # VALIDATION
    model.eval()
    all_p, all_y = [], []

    with torch.no_grad():
        for x, y in val_loader:
            out = model(x.to(DEVICE))
            all_p.extend(torch.argmax(out, 1).cpu().numpy())
            all_y.extend(y.numpy())

    f1 = f1_score(all_y, all_p, average='macro')

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": total_loss / len(train_loader),
        "val_f1": f1
    })

    print(f"Epoch {epoch+1} - Macro F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model_scratch.pth")

wandb.finish()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ W&B initialized in OFFLINE mode
Total files: 4000
Unique sample rates: {44100}
Average duration: 30.02404707482993
Class distribution: {'disco': 100, 'metal': 100, 'reggae': 100, 'blues': 100, 'rock': 100, 'classical': 100, 'jazz': 100, 'hiphop': 100, 'country': 100, 'pop': 100}
Total songs: 1000


Epoch 1: 100%|██████████| 57/57 [02:59<00:00,  3.16s/it]


Epoch 1 - Macro F1: 0.0535


Epoch 2: 100%|██████████| 57/57 [02:59<00:00,  3.15s/it]


...

Epoch 13 - Macro F1: 0.1044


Epoch 14: 100%|██████████| 57/57 [03:06<00:00,  3.27s/it]


Epoch 14 - Macro F1: 0.0727


Epoch 15: 100%|██████████| 57/57 [03:07<00:00,  3.30s/it]


Epoch 15 - Macro F1: 0.1159


In [3]:
# ======================
# TEST DATASET
# ======================
test_df = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))

class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.target = 22050 * 30

    def fix_len(self, y):
        if len(y) < self.target:
            y = np.pad(y, (0, self.target - len(y)))
        else:
            y = y[:self.target]
        return y

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(BASE_PATH, self.df.iloc[idx]["filename"])
        y, _ = librosa.load(path, sr=22050)
        y = self.fix_len(y)

        mel = librosa.feature.melspectrogram(
            y=y, sr=22050, n_mels=128, n_fft=2048, hop_length=512
        )
        mel = librosa.power_to_db(mel, ref=np.max)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        return torch.tensor(mel).unsqueeze(0).float()

# ======================
# TEST LOADER
# ======================
test_loader = DataLoader(TestDataset(test_df), batch_size=16, shuffle=False)

# ======================
# LOAD BEST MODEL
# ======================
model.load_state_dict(torch.load("best_model_scratch.pth", map_location=DEVICE))
model.eval()

# ======================
# PREDICTIONS
# ======================
preds = []

with torch.no_grad():
    for x in test_loader:
        x = x.to(DEVICE)
        out = model(x)
        preds.extend(torch.argmax(out, 1).cpu().numpy())

# ======================
# SUBMISSION
# ======================
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": [rev_label_map[p] for p in preds]
})

submission.to_csv("submission.csv", index=False)

print("✅ submission.csv created")
print(submission.head())

✅ submission.csv created
   id      genre
0   1      disco
1   2  classical
2   3      metal
3   4      metal
4   5    country


In [4]:
# # =========================
# # INSTALL
# # =========================
# !pip install transformers -q

# # =========================
# # IMPORTS
# # =========================
# import torch
# import torch.nn as nn
# import librosa
# import numpy as np
# import os
# import pandas as pd

# from torch.utils.data import Dataset, DataLoader
# from transformers import HubertModel
# from sklearn.metrics import f1_score

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # =========================
# # DATASET (TRAIN)
# # =========================
# class HubertDataset(Dataset):
    
#     def __init__(self, data, target_len=16000*15):  # 15 sec
#         self.data = data
#         self.target_len = target_len
    
#     def fix_length(self, y):
#         if len(y) < self.target_len:
#             y = np.pad(y, (0, self.target_len - len(y)))
#         else:
#             start = np.random.randint(0, len(y) - self.target_len)
#             y = y[start:start+self.target_len]
#         return y
    
#     def __len__(self):
#         return len(self.data)
    
#     def __getitem__(self, idx):
#         stems, label = self.data[idx]
        
#         audio = []
#         for s in stems.values():
#             y, _ = librosa.load(s, sr=16000)
#             y = self.fix_length(y)
#             audio.append(y)
        
#         mix = np.sum(audio, axis=0)
#         return torch.tensor(mix).float(), label


# # =========================
# # MODEL
# # =========================
# class HubertClassifier(nn.Module):
    
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
        
#         # FREEZE HUBERT
#         for param in self.hubert.parameters():
#             param.requires_grad = False
        
#         self.fc = nn.Sequential(
#             nn.Linear(self.hubert.config.hidden_size, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, num_classes)
#         )
    
#     def forward(self, x):
#         with torch.no_grad():
#             x = self.hubert(x).last_hidden_state
        
#         x = x.mean(dim=1)
#         return self.fc(x)


# # =========================
# # LOADERS
# # =========================
# train_loader = DataLoader(HubertDataset(train_data), batch_size=4, shuffle=True)
# val_loader = DataLoader(HubertDataset(val_data), batch_size=4)


# # =========================
# # TRAINING SETUP
# # =========================
# model = HubertClassifier(len(label_map)).to(device)

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

# EPOCHS = 5
# best_f1 = 0


# # =========================
# # TRAIN LOOP
# # =========================
# for epoch in range(EPOCHS):
    
#     model.train()
    
#     for x, y in train_loader:
#         x = x.to(device)
#         y = y.to(device)
        
#         optimizer.zero_grad()
#         out = model(x)
#         loss = criterion(out, y)
#         loss.backward()
#         optimizer.step()
    
#     # VALIDATION
#     model.eval()
#     preds, targets = [], []
    
#     with torch.no_grad():
#         for x, y in val_loader:
#             x = x.to(device)
#             out = model(x)
#             p = torch.argmax(out, 1).cpu().numpy()
            
#             preds.extend(p)
#             targets.extend(y.numpy())
    
#     f1 = f1_score(targets, preds, average="macro")
#     print(f"Epoch {epoch+1} F1:", f1)

#     if f1 > best_f1:
#         best_f1 = f1
#         torch.save(model.state_dict(), "best_model_hubert.pth")


# # =========================
# # TEST DATASET
# # =========================
# class TestDataset(Dataset):
    
#     def __init__(self, df, target_len=16000*15):
#         self.df = df
#         self.target_len = target_len
    
#     def fix_length(self, y):
#         if len(y) < self.target_len:
#             y = np.pad(y, (0, self.target_len - len(y)))
#         else:
#             y = y[:self.target_len]
#         return y
    
#     def __len__(self):
#         return len(self.df)
    
#     def __getitem__(self, idx):
        
#         path = BASE_PATH + "/" + self.df.iloc[idx]["filename"]
        
#         y, _ = librosa.load(path, sr=16000)
#         y = self.fix_length(y)
        
#         return torch.tensor(y).float()


# # =========================
# # TEST LOADER
# # =========================
# test_df = pd.read_csv(BASE_PATH + "/test.csv")

# test_loader = DataLoader(
#     TestDataset(test_df),
#     batch_size=4,
#     shuffle=False
# )


# # =========================
# # LOAD BEST MODEL
# # =========================
# model.load_state_dict(torch.load("best_model_hubert.pth", map_location=device))
# model.eval()


# # =========================
# # PREDICTION
# # =========================
# rev_label_map = {v:k for k,v in label_map.items()}

# predictions = []

# with torch.no_grad():
    
#     for x in test_loader:
        
#         x = x.to(device)
        
#         out = model(x)
        
#         pred = torch.argmax(out,1).cpu().numpy()
        
#         predictions.extend(pred)


# # =========================
# # SUBMISSION
# # =========================
# genres_pred = [rev_label_map[p] for p in predictions]

# submission = pd.DataFrame({
#     "id": test_df["id"],
#     "genre": genres_pred
# })

# submission.to_csv("submission.csv", index=False)

# print(submission.head())